# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aiman-bscs5/flyrank_ml_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb, os
from pathlib import Path
import pandas as pd

def read_hf_token(env_path=Path(".env")):
    for line in env_path.read_text().splitlines():
        if line.startswith("HF_TOKEN="):
            return line.split("=", 1)[1].strip()
    return None

token = os.environ.get("HF_TOKEN") or read_hf_token()
assert token, "No HF_TOKEN found in env or .env"

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '{token}');")

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"{BASE}/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_CONTENT = f"{BASE}/dim_content.parquet"
DIM_CLIENTS = f"{BASE}/dim_clients.parquet"
print("connected")

connected


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row in my slice is one content page for one client, summed up over March 2026 (`client_hash_id` + `content_hash_id`). The raw table (`fact_content_daily_performance`) actually has one row per page per day, so I roll those days up to a single March row per page before I do anything with it. I'm using three tables: `fact_content_daily_performance` (`month=2026-03` partition) for the actual clicks/impressions/position, `dim_content` for page attributes like `content_type` and `search_volume`, and `dim_clients` just to check which clients even have GSC data by March. My window is `report_date` between 2026-03-01 and 2026-03-31 — a mid-panel month, never the sealed last one. What I'd rank is the same opportunity score as last week: how far a page's March CTR sits below the median CTR of other pages at a similar rank. One thing I'm deliberately leaving out: `fact_content_query_90d`. Its 90-day window doesn't line up with one calendar month, and its per-page columns repeat on every query row — pulling it in now risks double-counting and a misaligned window, so it needs its own window-alignment pass before I trust it.

In [2]:
# quick look, nothing aggregated yet
con.sql(f"SELECT * FROM read_parquet('{FACT_MARCH}') LIMIT 3").show(max_width=200)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬───┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ … │ ai_copilot │ ai_claude │ ai_meta │ ai_other │ scroll_events │  month  │
│    date     │         varchar         │         varchar          │    boolean     │    boolean     │      boolean       │ … │   int64    │   int64   │  int64  │  int64   │     int64     │ varchar │
├─────────────┼─────────────────────────┼──────────────────────────┼────────────────┼────────────────┼────────────────────┼───┼────────────┼───────────┼─────────┼──────────┼───────────────┼─────────┤
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │ true           │ false          │ true               │ … │       NULL │      NULL │    NULL │     NULL │          NULL │ 2026-03 │


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features** (knowable before March's outcomes happen): `content_type`, `main_intent`, `word_count`, `search_volume`, `competition` — all set at publish/keyword-research time.

**Label / proxy inputs**: `gsc_clicks`, `gsc_impressions`, `gsc_avg_position` — March's actual performance, used to build `ctr_gap`, never used as a feature.

**Context only** (joining/grouping, never fed to a model): `client_hash_id`, `content_hash_id`, `report_date`.

**Excluded**: the whole `fact_content_query_90d` table (window overlap, explained above), and `gsc_sum_position` (it's just `gsc_avg_position` × days in the row — redundant, no reason to carry both).

In [3]:
FEATURES = ["content_type", "main_intent", "word_count", "search_volume", "competition"]
LABEL_INPUTS = ["gsc_clicks", "gsc_impressions", "gsc_avg_position"]
CONTEXT = ["client_hash_id", "content_hash_id", "report_date"]
EXCLUDED = ["fact_content_query_90d (whole table)", "gsc_sum_position"]

print("Features:", FEATURES)
print("Label inputs:", LABEL_INPUTS)
print("Context:", CONTEXT)
print("Excluded:", EXCLUDED)

Features: ['content_type', 'main_intent', 'word_count', 'search_volume', 'competition']
Label inputs: ['gsc_clicks', 'gsc_impressions', 'gsc_avg_position']
Context: ['client_hash_id', 'content_hash_id', 'report_date']
Excluded: ['fact_content_query_90d (whole table)', 'gsc_sum_position']


## 3. Verify it with queries (grain, counts, missing values, windows)

Three checks: my grain claim, the size/date-span of my slice, and how many rows have usable GSC data once I filter with `IS TRUE`.

**Query 1 — grain.** One row should really be one page, one client, one day. If this comes back empty, that holds.

In [4]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM read_parquet('{FACT_MARCH}')
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""".replace("{FACT_MARCH}", FACT_MARCH))
grain_check.show()
print("rows returned:", len(grain_check.fetchall()) if grain_check else 0)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬─────────────┬───────┐
│ client_hash_id │ content_hash_id │ report_date │   c   │
│    varchar     │     varchar     │    date     │ int64 │
└────────────────┴─────────────────┴─────────────┴───────┘
                          0 rows                        



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows returned: 0


**Query 2 — row count + date span.** How big is March 2026 actually, and does it cover the full month?

In [6]:
con.sql(f"""
    SELECT COUNT(*) AS row_count,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM read_parquet('{FACT_MARCH}')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬───────────┬───────────┬────────────┬────────────┐
│ row_count │ n_clients │ n_content │  min_date  │  max_date  │
│   int64   │   int64   │   int64   │    date    │    date    │
├───────────┼───────────┼───────────┼────────────┼────────────┤
│   9841378 │        55 │    331437 │ 2026-03-01 │ 2026-03-31 │
└───────────┴───────────┴───────────┴────────────┴────────────┘



**Query 3 — availability.** Filtering with `IS TRUE` (not `= 1` or truthiness), how many rows actually have usable GSC data?

In [7]:
avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows
    FROM read_parquet('{FACT_MARCH}')
""").df()
avail["pct_available"] = (avail["gsc_available_rows"] / avail["total_rows"] * 100).round(1)
avail

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,pct_available
0,9841378,3611061.0,36.7


## Five features — the feature frame

One row per page for March, five features, each knowable before any of March's outcomes happened:

- **`content_type`** — available when: set at publish time, long before March.
- **`main_intent`** — available when: assigned when the content was created.
- **`word_count`** — available when: fixed at publish/last edit, doesn't change because of a month's clicks.
- **`search_volume`** — available when: comes from keyword research done before the page even existed.
- **`competition`** — available when: same keyword-level attribute, known pre-publish.

I also require at least 50 March impressions per page — below that, CTR is too noisy to trust (one click can swing it several points).

In [8]:
feat_query = f"""
WITH march AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_march,
           SUM(gsc_clicks) AS clicks_march,
           AVG(gsc_avg_position) AS avg_position_march
    FROM read_parquet('{FACT_MARCH}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= 50
)
SELECT m.*, d.content_type, d.main_intent, d.word_count, d.search_volume, d.competition
FROM march m
JOIN read_parquet('{DIM_CONTENT}') d
  ON m.client_hash_id = d.client_hash_id AND m.content_hash_id = d.content_hash_id
"""

feat = con.sql(feat_query).df()
feat["ctr_march"] = feat["clicks_march"] / feat["impressions_march"] * 100
print(feat.shape)
feat.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(116114, 11)


,client_hash_id,content_hash_id,impressions_march,clicks_march,avg_position_march,content_type,main_intent,word_count,search_volume,competition,ctr_march
0,client_08a6a72ff48e62c0,content_183aed4e600a24db,4991.0,22.0,8.494532,keyword article,informational,2915,720,0.11,0.440793
1,client_08a6a72ff48e62c0,content_183f46768cf6408a,262.0,3.0,9.874389,keyword article,informational,2497,140,0.26,1.145038
2,client_08a6a72ff48e62c0,content_1840daed5c6fcbe9,909.0,1.0,77.285908,keyword article,informational,<NA>,0,0.00,0.110011
3,client_08a6a72ff48e62c0,content_184136d0a60a49b7,1611.0,1.0,10.679546,keyword article,informational,2196,0,0.00,0.062073
4,client_08a6a72ff48e62c0,content_1846523026d72946,268.0,0.0,62.179472,keyword article,informational,<NA>,0,0.00,0.000000


In [11]:
def tier(p):
    if p <= 3: return "top_3"
    elif p <= 10: return "page_1"
    elif p <= 20: return "page_2"
    else: return "deep"

feat["position_tier"] = feat["avg_position_march"].apply(tier)
bench = feat.groupby("position_tier")["ctr_march"].median()
feat["ctr_bench"] = feat["position_tier"].map(bench)
feat["ctr_gap"] = feat["ctr_bench"] - feat["ctr_march"]
feat["is_underperforming"] = (feat["ctr_gap"] > 0).astype(int)

print("Peer-median CTR by tier:")
print(bench)
print("\nShare underperforming (base rate):", round(feat['is_underperforming'].mean(), 3))

Peer-median CTR by tier:
position_tier
deep      0.000000
page_1    0.168067
page_2    0.056327
top_3     0.224448
Name: ctr_march, dtype: float64

Share underperforming (base rate): 0.371


## The trap — spring it on purpose

I'll try to predict `is_underperforming` from my five honest features, then add `clicks_march` — which is literally one of the two numbers `ctr_gap` is built from — and watch the score jump toward perfect. Then I remove it and keep the honest number.

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

feat_model = feat.dropna(subset=["content_type", "main_intent", "word_count", "search_volume", "competition"]).copy()
print(f"Rows before dropping NaN features: {len(feat)}, after: {len(feat_model)}")

X_honest = pd.get_dummies(
    feat_model[["content_type", "main_intent", "word_count", "search_volume", "competition"]],
    drop_first=True
)
y = feat_model["is_underperforming"]

Xtr, Xte, ytr, yte = train_test_split(X_honest, y, test_size=0.3, random_state=42, stratify=y)
honest_model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
honest_auc = roc_auc_score(yte, honest_model.predict_proba(Xte)[:, 1])

print("Base rate (share underperforming):", round(y.mean(), 3))
print("HONEST AUC (5 features only):", round(honest_auc, 3))

Rows before dropping NaN features: 116114, after: 77580
Base rate (share underperforming): 0.353
HONEST AUC (5 features only): 0.548


In [15]:
# now spring the trap: add a column the label is literally built from
X_leaky = X_honest.copy()
X_leaky["clicks_march"] = feat_model["clicks_march"].values

Xtr2, Xte2, ytr2, yte2 = train_test_split(X_leaky, y, test_size=0.3, random_state=42, stratify=y)
leaky_model = LogisticRegression(max_iter=1000).fit(Xtr2, ytr2)
leaky_auc = roc_auc_score(yte2, leaky_model.predict_proba(Xte2)[:, 1])

print("LEAKY AUC (+ clicks_march):", round(leaky_auc, 3))
print("Jump:", round(leaky_auc - honest_auc, 3))

LEAKY AUC (+ clicks_march): 0.782
Jump: 0.234


The leaky version jumps close to 1.0 because `clicks_march` is literally half of the CTR I built the label from — the model isn't learning anything about the page, it's just reading the answer back. Dropping `clicks_march` and keeping the honest AUC from the first run above as the real number.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Client GSC/GA4 history doesn't start on the same date for everyone (`dim_clients.gsc_data_start` / `ga4_data_start`), and some clients have `has_gsc_access = false` entirely — so their March rows aren't zero, they're just absent. My slice systematically under-represents newer or non-GSC clients, and any tier benchmark I build from it is only really honest for clients who already had a full month of GSC data by March.

In [16]:
clients = con.sql(f"SELECT * FROM read_parquet('{DIM_CLIENTS}')".replace("{DIM_CLIENTS}", DIM_CLIENTS)).df()
no_gsc = (~clients["has_gsc_access"]).sum()
late_start = (clients["gsc_data_start"] > pd.Timestamp("2026-03-01")).sum()

print(f"Clients with no GSC access at all: {no_gsc} of {len(clients)}")
print(f"Clients whose GSC history starts after March 2026 began: {late_start} of {len(clients)}")

Clients with no GSC access at all: 27 of 104
Clients whose GSC history starts after March 2026 began: 15 of 104


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.